# Day 11 — Solutions: Debugging, Logging, Profiling
Logging instrumentation and micro-benchmarks with timeit/cProfile.

In [ ]:
import logging, json, csv
from pathlib import Path
from typing import Any

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
log = logging.getLogger(__name__)

def safe_load_json(path: str | Path) -> Any | None:
    p = Path(path)
    log.info('loading JSON: %s', p)
    try:
        return json.loads(p.read_text(encoding='utf-8'))
    except FileNotFoundError:
        log.error('not found: %s', p)
    except PermissionError:
        log.error('permission denied: %s', p)
    except json.JSONDecodeError as e:
        log.error('invalid JSON %s (line %d col %d): %s', p, e.lineno, e.colno, e.msg)
    return None

safe_load_json('missing.json')

In [ ]:
# Profiling example: vectorization vs Python loop
import timeit, numpy as np

def scale_loop(xs: list[float], k: float) -> list[float]:
    out = []
    for x in xs:
        out.append(x * k)
    return out

def scale_vec(xs: np.ndarray, k: float) -> np.ndarray:
    return xs * k

arr = np.arange(100_000, dtype=float)
lst = arr.tolist()
loop_t = timeit.timeit(lambda: scale_loop(lst, 2.0), number=5)
vec_t  = timeit.timeit(lambda: scale_vec(arr, 2.0), number=5)
{'loop_sec': loop_t, 'vec_sec': vec_t}